# Building nodes with `client.nodes`

The Uncertainty Engine has many nodes, and users can register their own. Rather than a hand-written Python class per node (which has to be kept in step with the registry and can never cover nodes users create), the SDK builds nodes **at runtime** from the registry.

`client.nodes` is how you do that. It looks up only the node you ask for, when you ask for it, and it works for any node the registry knows about, including ones the SDK has never heard of.

This notebook covers finding nodes, inspecting them, building them, versions, wiring them into a workflow, and how their schemas are cached.

## Setup

In [ ]:
from uncertainty_engine import Client

client = Client()
client.authenticate()

## Discovering nodes

There is nothing to import for a node. Ask the registry which nodes exist with `available()`, which returns their names, sorted.

In [2]:
names = client.nodes.available()

print(len(names), "nodes available")
names[:15]

73 nodes available


['Add',
 'AnalyseVariance',
 'AppendDataset',
 'BasicMathsOperators',
 'BuildSensorDesigner',
 'ComputeImplausibility',
 'ComputeLikelihood',
 'CoralScop',
 'CreateChat',
 'CreateDatasetSlice',
 'CreateFEM',
 'CreateInitialDesign',
 'CreateSurrogateSamples',
 'Dict',
 'Display']

`list_nodes()` returns the full catalogue entry for each node, so you can, for example, browse the nodes by category.

In [3]:
from collections import defaultdict

by_category = defaultdict(list)

for node in client.list_nodes():
    by_category[node["category"]].append(node["id"])

for category in sorted(by_category):
    print(f"{category}: {', '.join(sorted(by_category[category]))}")

Algorithm: HoughTransform, KWP6Postprocess, PcdProcessing
Calibration: CreateSurrogateSamples, ProjectileSimulator, RecommendNROYSamples, TokomakPlayground
Control Logic: Join, Splitter, SupervisorAdvice
Datasets: AppendDataset, CreateDatasetSlice, FilterDataset, Split
Inputs: Dict, Load, LoadChatHistory, LoadDataset, LoadDocument, LoadModel, LoadMultiple, LoadTransform, Message, Number, Text
Mathematical Operators: Add, AnalyseVariance, BasicMathsOperators, ComputeImplausibility, ComputeLikelihood, MaxLikelihood, SummariseSampleStatistics
MathematicalOperators: Max
Model: CoralScop, SemanticObjectDetection, SemanticObjectSegmentation, StereoReconstruction
Models: GetContext, LLMConfig, ModelConfig, PredictModel, PredictPosteriorConditioning, PromptLLM, ScoreModel, TrainModel
Other: CreateChat
Outputs: CreateFEM, Display, Download, ExportTorchScript, ParityPlot, PlotLossCurve, PlotSamples, Render, Save, UncertaintyPlot
Samplers: CreateInitialDesign, MonteCarloSampler, RunMcmc, RunSampl

In Jupyter or IPython you can also type `client.nodes.` and press **Tab** to autocomplete node names.

## Inspecting a node

`describe()` returns a node's schema from the registry: its description, and the name, type and description of each input and output. This is how you find out what a node needs.

In [4]:
info = client.nodes.describe("Add")

print(info.description)
print()
print("Inputs:")
for name, spec in info.inputs.items():
    print(f"  {name}: {spec.type} (required={spec.required}) - {spec.description}")
print("Outputs:")
for name, spec in info.outputs.items():
    print(f"  {name}: {spec.type} - {spec.description}")

Add two numbers and return the sum.

Inputs:
  lhs: float (required=True) - Left-hand side of the addition
  rhs: float (required=True) - Right-hand side of the addition
Outputs:
  ans: float - Result of the addition


A node can have several versions registered. `get_node_versions()` lists them.

In [5]:
client.get_node_versions("Add")

['0.3.0', 'latest', '0.2.0']

## Building a node

Build a node by calling it as an attribute of `client.nodes`, passing its inputs and a label. The result is the payload the Uncertainty Engine receives: the node's name and its inputs.

In [6]:
add = client.nodes.Add(lhs=1, rhs=2, label="add")

add()

('Add', {'lhs': 1, 'rhs': 2})

If the node's name is in a variable, call `client.nodes` with the name instead.

In [7]:
node_name = "Add"

add_again = client.nodes(node_name, lhs=3, rhs=4, label="add_again")

add_again()

('Add', {'lhs': 3, 'rhs': 4})

Every registered node works the same way, including nodes the SDK never had a class for, such as `Number`.

In [8]:
number = client.nodes.Number(value=5, label="number")

number()

('Number', {'value': 5})

Built nodes are ordinary `Node` objects, so everything else in the SDK, such as running them or adding them to a graph, works as before.

In [9]:
type(add)

uncertainty_engine.nodes.base.Node

## Errors as you build

Inputs are checked against the node's schema when you build it, so a missing or unknown input fails straight away rather than when the node runs.

In [10]:
from uncertainty_engine.exceptions import NodeValidationError

try:
    client.nodes.Add(lhs=1, label="missing_rhs")
except NodeValidationError as error:
    print(error)

try:
    client.nodes.Add(lhs=1, rhs=2, not_an_input=3, label="unknown_input")
except NodeValidationError as error:
    print(error)

Missing required inputs: ['rhs']
Invalid input names: ['not_an_input']


A node the registry doesn't know raises a `NodeNotFoundError`, which points you at `available()`.

In [11]:
from uncertainty_engine.exceptions import NodeNotFoundError

try:
    client.nodes.NotARealNode(label="nope")
except NodeNotFoundError as error:
    print(error)

Node 'NotARealNode' was not found. Use `client.nodes.available()` to see the available nodes.


## Versions

By default a node is built at the version the registry considers its default. Pass `version` to build a specific one.

In [12]:
default_add = client.nodes.Add(lhs=1, rhs=2, label="default_add")
pinned_add = client.nodes.Add(lhs=1, rhs=2, label="pinned_add", version="0.2.0")

print("default:", default_add.version)
print("pinned: ", pinned_add.version)

default: latest
pinned:  0.2.0


To pin versions for several nodes at once, make a pinned view with `with_versions()`. Nodes in the mapping are built at the version given; every other node still uses its default. `client.nodes` itself is left unchanged.

In [13]:
pinned = client.nodes.with_versions({"Add": "0.2.0"})

print("Add from the pinned view:   ", pinned.Add(lhs=1, rhs=2, label="a").version)
print("Number from the pinned view:", pinned.Number(value=1, label="n").version)
print("Add from client.nodes:      ", client.nodes.Add(lhs=1, rhs=2, label="b").version)

Add from the pinned view:    0.2.0
Number from the pinned view: latest
Add from client.nodes:       latest


## Wiring nodes into a workflow

Connect nodes by passing one node's output handle as another node's input, then add them to a `Graph` and run it as a workflow, exactly as with any other nodes. See [`workflow.ipynb`](./workflow.ipynb) for more on workflows.

In [14]:
from uncertainty_engine.graph import Graph

graph = Graph()

five = client.nodes.Number(value=5, label="Five")
add_ten = client.nodes.Add(lhs=five.make_handle("value"), rhs=10, label="Add Ten")

graph.add_node(five)
graph.add_node(add_ten)

In [15]:
from pprint import pprint

from uncertainty_engine.nodes.workflow import Workflow

workflow = Workflow(
    graph=graph.nodes,
    inputs=graph.external_input,
    external_input_id=graph.external_input_id,
    requested_output={
        "Result": add_ten.make_handle("ans").model_dump(),
    },
    client=client,
)

response = client.run_node(workflow)
pprint(response.model_dump())

{'inputs': {'_nodes_list_error': None,
            'external_input_id': '_',
            'graph': {'nodes': {'Add Ten': {'inputs': {'lhs': {'node_handle': 'value',
                                                               'node_name': 'Five'},
                                                       'rhs': {'node_handle': 'Add '
                                                                              'Ten_rhs',
                                                               'node_name': '_'}},
                                            'type': 'Add',
                                            'version': 'latest'},
                                'Five': {'inputs': {'value': {'node_handle': 'Five_value',
                                                              'node_name': '_'}},
                                         'type': 'Number',
                                         'version': 'latest'}}},
            'inputs': {'Add Ten_rhs': 10, 'Five_value': 5},
            

## Inputs with structured types

`client.nodes` passes inputs through exactly as the node declares them. Most inputs are plain values or handles, but a few nodes take a structured value. For example, the nodes that load a resource take its ID as `{"id": ...}`, and `Save` takes the resource name as `file_id`.

The cell below only builds the nodes: replace the placeholders with your own project and resource IDs to run them. See [`train_predict.ipynb`](./train_predict.ipynb) for a full example.

In [16]:
load_dataset = client.nodes.LoadDataset(
    label="Load Dataset",
    project_id="<your-project-id>",
    file_id={"id": "<your-dataset-resource-id>"},  # a resource ID is passed as {"id": ...}
)

save = client.nodes.Save(
    label="Save Dataset",
    data=load_dataset.make_handle("file"),
    project_id="<your-project-id>",
    file_id="my-saved-dataset",  # the name to save the resource under
)

load_dataset()

('LoadDataset',
 {'project_id': '<your-project-id>',
  'file_id': {'id': '<your-dataset-resource-id>'}})

## Caching

The first time a node is built, its schema is fetched from the registry. After that the client keeps it, so building the same node again, from `client.nodes` or from any pinned view, makes no further request. The catalogue from `available()` and `list_nodes()` is cached the same way.

The cache lasts for the life of the client. To pick up a node or version deployed since, clear it.

In [17]:
# Add's schema was fetched above, so this makes no request
client.nodes.Add(lhs=1, rhs=2, label="cached")

# Forget every cached schema and the catalogue
client.clear_node_cache()

# This fetches Add's schema again
client.nodes.Add(lhs=1, rhs=2, label="fresh")

## Summary

- **`client.nodes.available()`** lists every node name, and **`client.nodes.describe("X")`** shows how to call a node.
- **`client.nodes.X(...)`** or **`client.nodes("X", ...)`** builds a node, checking its inputs against the schema.
- **`version=`** builds a specific version, and **`with_versions()`** pins several at once.
- **`client.clear_node_cache()`** picks up newly deployed nodes and versions.

Because node names only exist in the registry at runtime, static editors and type checkers can't autocomplete them; use tab-completion in Jupyter, `available()` and `describe()` instead.